# Week 11: DBSCAN, Hierarchical Agglomerative Clustering (HAC), Linkage Methods, and Dendrograms

## 1. Notebook Setup
- Import libraries

In [ ]:
#imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.cluster import DBSCAN, AgglomerativeClustering
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import dendrogram, linkage


plt.style.use("seaborn-v0_8-whitegrid")
sns.set_context("notebook", font_scale=1.1)

## 2. Dataset 1: customer_churn
### 2.1 Data Overview & Preparation

In [ ]:
#load data
train_cc = pd.read_csv('customer_churn_dataset-training-master.csv')
test_cc = pd.read_csv('customer_churn_dataset-testing-master.csv')
#combine train and test set
customer_churn = pd.concat([train_cc, test_cc], ignore_index=True)
#drop null rows
customer_churn.dropna(inplace=True)
#drop CustomerID col (irrelevant)
customer_churn = customer_churn.drop('CustomerID', axis=1)
#use sample of dataset for quicker runtime
sample, _ = train_test_split(
    customer_churn,
    train_size = 0.3, #use 30% of dataset
    stratify = customer_churn['Churn'],
    random_state = 42
)

#preprocessing: remove target cols and scale features
target_col = "Churn"
def preprocess(df, target_col):
    X = pd.get_dummies(df.drop(target_col, axis=1), drop_first=True)
    y = df[target_col]
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    return X, X_scaled, y
X_cc, X_cc_scaled, y_cc = preprocess(sample, target_col)

### 2.2 Dimensionality Reduction (PCA for visualization)

In [ ]:
def compute_pca(X_scaled):
    pca = PCA(n_components=2)
    pcs = pca.fit_transform(X_scaled)
    return pcs
pcs_cc = compute_pca(X_cc_scaled)
print(pcs_cc)

### 2.3: DBSCAN Clustering

In [ ]:
def run_dbscan(X_scaled, pcs, title, eps_val=0.5, min_samples_val=5):
    db = DBSCAN(eps=eps_val, min_samples=min_samples_val)
    labels = db.fit_predict(X_scaled)


    plt.figure(figsize=(7,5))
    plt.scatter(pcs[:,0], pcs[:,1], c=labels, cmap='tab20', s=15)
    plt.title(f'DBSCAN Clustering – {title}\n(eps={eps_val}, min_samples={min_samples_val})')
    plt.xlabel('PC1')
    plt.ylabel('PC2')
    plt.show()
run_dbscan(X_cc_scaled, pcs_cc, "Customer Churn")

### 2.4: Hierarchical Agglomerative Clustering (HAC)

In [ ]:
def run_hac(X_scaled, pcs, title, linkage_method="ward", n_clusters=3):
    hac = AgglomerativeClustering(n_clusters=n_clusters, linkage=linkage_method)
    labels = hac.fit_predict(X_scaled)


    plt.figure(figsize=(7,5))
    plt.scatter(pcs[:,0], pcs[:,1], c=labels, cmap='viridis', s=15)
    plt.title(f'HAC ({linkage_method}) – {title}')
    plt.xlabel('PC1')
    plt.ylabel('PC2')
    plt.show()
    #run HAC for 3 linkage methods
for method in ["ward", "complete", "average"]:    
    run_hac(X_cc_scaled, pcs_cc, f"Customer Churn ({method})", linkage_method=method)

### 2.5: Dendrograms for HAC

In [ ]:
def plot_dendrogram(X_scaled, title, linkage_method="ward"):
    Z = linkage(X_scaled, method=linkage_method)
    plt.figure(figsize=(10, 5))
    dendrogram(Z, truncate_mode='level', p=5)
    plt.title(f'Dendrogram – {title} ({linkage_method} linkage)')
    plt.xlabel('Sample Index or Cluster Size')
    plt.ylabel('Distance')
    plt.show()
plot_dendrogram(X_cc_scaled, "Customer Churn", "ward")

#### Additional plots:
[insert any plots]

## 3. Dataset 2: digital_marketing_campaign

### 3.1: Data Overview & Preparation 

In [ ]:
digital_marketing = pd.read_csv("digital_marketing_campaign_dataset.csv")
X = pd.get_dummies(digital_marketing.drop("Conversion", axis=1, errors="ignore"), drop_first=True)
y = digital_marketing["Conversion"] if "Conversion" in digital_marketing.columns else None

X_dm, X_dm_scaled, y_dm = preprocess(digital_marketing, 'Response')

### 3.2: Dimensionality Reduction (PCA for visualization)

In [ ]:
pcs_dm = compute_pca(X_dm_scaled)
print(pcs_dm)

### 3.3: DBSCAN Clustering

In [ ]:
run_dbscan(X_dm_scaled, pcs_dm, "Digital Marketing")

### 3.4: Hierarchical Agglomerative Clustering (HAC)

In [ ]:
for method in ["ward", "complete", "average"]:
    run_hac(X_dm_scaled, pcs_dm, f"Digital Marketing ({method})", linkage_method=method)

### 3.5: Dendrograms for HAC

In [ ]:
plot_dendrogram(X_dm_scaled, "Digital Marketing", "complete")

## 4. Dataset 3: marketing_campaign

### 4.1: Data Overview & Preparation 

In [ ]:
marketing_campaign = pd.read_csv("marketing_campaign.csv", sep=';')
X = pd.get_dummies(marketing_campaign.drop("Response", axis=1, errors="ignore"), drop_first=True)
y = marketing_campaign["Response"] if "Response" in marketing_campaign.columns else None

X_mc, X_mc_scaled, y_mc = preprocess(marketing_campaign, 'Success')

### 4.2: Dimensionality Reduction (PCA for visualization)

In [ ]:
pcs_mc = compute_pca(X_mc_scaled)
print(pcs_mc)

### 4.3: DBSCAN Clustering

In [ ]:
run_dbscan(X_mc_scaled, pcs_mc, "Marketing Campaign")

### 4.4: PCA Variance Explained Plots

In [ ]:
for method in ["ward", "complete", "average"]:
    run_hac(X_mc_scaled, pcs_mc, f"Marketing Campaign ({method})", linkage_method=method)

### 4.5: Dendrograms for HAC

plot_dendrogram(X_mc_scaled, "Marketing Campaign", "average")